# Kaggle Submission Exp 008 Long Context Hybrid

This notebook submits the first long-context native branch:
- `exp_008` long-context model checkpoint
- `exp_008b` winning postprocess recipe
- single-fold native hybrid with `event + texture priors + smoothing`

Expected Kaggle inputs:
- BirdCLEF+ 2026 competition data
- one dataset containing `fold_00/best_model.pt` from `exp_008`

Recommended runtime:
- `Internet = Off`
- `Accelerator = GPU`


In [ ]:

import os
import re
import time
import warnings
from contextlib import nullcontext
from dataclasses import dataclass
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import timm
import torchaudio.transforms as T

warnings.filterwarnings('ignore')
START = time.time()

if torch.cuda.is_available():
    device = torch.device('cuda')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

if device.type == 'cpu':
    torch.set_num_threads(max(1, os.cpu_count() or 1))

if hasattr(torch, 'set_float32_matmul_precision'):
    torch.set_float32_matmul_precision('high')

print('Torch:', torch.__version__)
print('Device:', device)
if device.type == 'cpu':
    print('CPU threads:', torch.get_num_threads())


In [ ]:

MODEL_DATASET_HINT = None  # Example: 'birdclef-exp008-long-context-fold0'
CHECKPOINT_NAME = 'best_model.pt'
FOLD_NAME = 'fold_00'
LAMBDA_EVENT = 0.4
LAMBDA_TEXTURE = 1.0
SMOOTH_TEXTURE = 0.35
TEXTURE_TAXA = {'Amphibia', 'Insecta'}
AUDIO_SUFFIXES = {'.ogg', '.wav', '.flac', '.mp3'}


@dataclass
class Config:
    sr: int = 32_000
    context_duration: float = 20.0
    target_duration: float = 5.0
    context_hop: float = 5.0
    n_mels: int = 224
    n_fft: int = 2048
    hop_length: int = 512
    fmin: int = 0
    fmax: int = 16_000
    top_db: float = 80.0
    power: float = 2.0
    norm: str = 'slaney'
    mel_scale: str = 'htk'
    backbone: str = 'tf_efficientnet_b0.ns_jft_in1k'
    num_classes: int = 234
    in_channels: int = 3
    dropout: float = 0.1
    drop_path_rate: float = 0.0
    gem_p_init: float = 3.0
    batch_contexts: int = 8

    @property
    def context_samples(self) -> int:
        return int(self.sr * self.context_duration)

    @property
    def target_samples(self) -> int:
        return int(self.sr * self.target_duration)

    @property
    def num_output_steps(self) -> int:
        return int(round(self.context_duration / self.target_duration))


cfg = Config()


def resolve_project_root() -> Path | None:
    for candidate in [Path.cwd(), Path.cwd().parent]:
        if (candidate / 'data' / 'birdclef-2026').exists():
            return candidate
    return None


PROJECT_ROOT = resolve_project_root()
INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else (PROJECT_ROOT / 'submissions' / 'debug' if PROJECT_ROOT is not None else Path.cwd())
WORK_ROOT.mkdir(parents=True, exist_ok=True)
SUBMISSION_PATH = WORK_ROOT / 'submission.csv'


def resolve_data_root() -> Path:
    direct_candidates = [
        INPUT_ROOT / 'competitions' / 'birdclef-2026',
        INPUT_ROOT / 'birdclef-2026',
    ]
    for candidate in direct_candidates:
        if (candidate / 'sample_submission.csv').exists():
            return candidate

    if INPUT_ROOT.exists():
        for child in sorted(INPUT_ROOT.iterdir()):
            if (child / 'sample_submission.csv').exists():
                return child
            matches = sorted(child.rglob('sample_submission.csv'))
            if matches:
                return matches[0].parent

    if PROJECT_ROOT is not None:
        local_candidate = PROJECT_ROOT / 'data' / 'birdclef-2026'
        if (local_candidate / 'sample_submission.csv').exists():
            return local_candidate

    raise FileNotFoundError('BirdCLEF competition data was not found under /kaggle/input or the local project data directory.')


def iter_candidate_roots(model_dataset_hint: str | None):
    if model_dataset_hint:
        hinted = Path(model_dataset_hint)
        if hinted.exists():
            yield hinted
        hinted_input = INPUT_ROOT / model_dataset_hint
        if hinted_input.exists():
            yield hinted_input

    if INPUT_ROOT.exists():
        for child in sorted(INPUT_ROOT.iterdir()):
            yield child

    if PROJECT_ROOT is not None:
        yield PROJECT_ROOT / 'experiments' / 'outputs' / 'exp_008_long_context_native_sed'
        yield PROJECT_ROOT / 'experiments' / 'outputs' / 'exp_008_long_context_native_sed' / FOLD_NAME


def resolve_checkpoint_path(model_dataset_hint: str | None = None, checkpoint_name: str = CHECKPOINT_NAME, fold_name: str = FOLD_NAME) -> Path:
    searched = []
    seen = set()
    candidates = []

    for root in iter_candidate_roots(model_dataset_hint):
        try:
            root = root.resolve()
        except FileNotFoundError:
            continue
        key = str(root)
        if key in seen or not root.exists():
            continue
        seen.add(key)
        searched.append(key)

        if root.is_file() and root.name == checkpoint_name:
            candidates.append(root)
            continue

        if root.is_dir():
            candidates.extend(sorted(root.rglob(checkpoint_name)))

    if not candidates:
        raise FileNotFoundError(f'Could not find {checkpoint_name} under candidate roots: {searched[:12]}')

    preferred = [path for path in candidates if fold_name in path.parts]
    chosen = preferred[0] if preferred else candidates[0]
    return chosen


DATA_ROOT = resolve_data_root()
CHECKPOINT_PATH = resolve_checkpoint_path(MODEL_DATASET_HINT, CHECKPOINT_NAME, FOLD_NAME)
print({
    'project_root': str(PROJECT_ROOT) if PROJECT_ROOT is not None else None,
    'data_root': str(DATA_ROOT),
    'checkpoint_path': str(CHECKPOINT_PATH),
    'submission_path': str(SUBMISSION_PATH),
    'lambda_event': LAMBDA_EVENT,
    'lambda_texture': LAMBDA_TEXTURE,
    'smooth_texture': SMOOTH_TEXTURE,
})


In [ ]:

SAMPLE_SUB = pd.read_csv(DATA_ROOT / 'sample_submission.csv')
SOUNDSCAPE_LABELS = pd.read_csv(DATA_ROOT / 'train_soundscapes_labels.csv')
TAXONOMY = pd.read_csv(DATA_ROOT / 'taxonomy.csv')
SPECIES = SAMPLE_SUB.columns[1:].tolist()
LABEL_TO_IDX = {label: idx for idx, label in enumerate(SPECIES)}


def parse_soundscape_labels(value):
    if pd.isna(value):
        return []
    return [token.strip() for token in str(value).split(';') if token.strip()]


def union_labels(series: pd.Series):
    merged = []
    seen = set()
    for value in series:
        for label in parse_soundscape_labels(value):
            if label not in seen:
                merged.append(label)
                seen.add(label)
    return merged


def parse_soundscape_key(name_or_stem: str):
    stem = Path(name_or_stem).stem
    parts = stem.split('_')
    site = parts[3] if len(parts) >= 4 else None
    time_token = parts[-1] if parts else '000000'
    hour_utc = int(time_token[:2]) if time_token.isdigit() and len(time_token) >= 2 else -1
    return site, hour_utc


def encode_multi_hot(labels, species, label_to_idx):
    target = np.zeros(len(species), dtype=np.float32)
    for label in labels:
        idx = label_to_idx.get(label)
        if idx is not None:
            target[idx] = 1.0
    return target


SC_CLEAN = (
    SOUNDSCAPE_LABELS
    .groupby(['filename', 'start', 'end'])['primary_label']
    .apply(union_labels)
    .reset_index(name='label_list')
)
SC_CLEAN['end_sec'] = pd.to_timedelta(SC_CLEAN['end']).dt.total_seconds().astype(int)
SC_CLEAN[['site', 'hour_utc']] = SC_CLEAN['filename'].apply(lambda x: pd.Series(parse_soundscape_key(x)))

TEXTURE_LABELS = TAXONOMY.loc[TAXONOMY['class_name'].isin(TEXTURE_TAXA), 'primary_label'].tolist()
TEXTURE_SET = set(TEXTURE_LABELS)
IDX_TEXTURE = np.array([LABEL_TO_IDX[label] for label in SPECIES if label in TEXTURE_SET], dtype=np.int32)
IDX_EVENT = np.array([idx for idx, label in enumerate(SPECIES) if label not in TEXTURE_SET], dtype=np.int32)

PRIOR_TARGETS = np.stack([encode_multi_hot(labels, SPECIES, LABEL_TO_IDX) for labels in SC_CLEAN['label_list']], axis=0)
print({
    'num_classes': len(SPECIES),
    'labeled_soundscape_segments': int(len(SC_CLEAN)),
    'texture_classes': int(len(IDX_TEXTURE)),
    'event_classes': int(len(IDX_EVENT)),
})


In [ ]:

def stable_logit(p: np.ndarray, eps: float = 1e-4) -> np.ndarray:
    p = np.clip(np.asarray(p, dtype=np.float32), eps, 1.0 - eps)
    return (np.log(p) - np.log1p(-p)).astype(np.float32, copy=False)


def stable_sigmoid(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    positive = x >= 0
    out = np.empty_like(x, dtype=np.float32)
    out[positive] = 1.0 / (1.0 + np.exp(-x[positive]))
    exp_x = np.exp(x[~positive])
    out[~positive] = exp_x / (1.0 + exp_x)
    return out


def fit_prior_tables(prior_df: pd.DataFrame, y_prior: np.ndarray):
    prior_df = prior_df.reset_index(drop=True)
    global_p = y_prior.mean(axis=0).astype(np.float32)

    site_keys = sorted(prior_df['site'].dropna().astype(str).unique().tolist())
    site_to_i = {key: idx for idx, key in enumerate(site_keys)}
    site_n = np.zeros(len(site_keys), dtype=np.float32)
    site_p = np.zeros((len(site_keys), y_prior.shape[1]), dtype=np.float32)
    site_series = prior_df['site'].astype(str)
    for site in site_keys:
        idx = site_to_i[site]
        mask = site_series.values == site
        site_n[idx] = mask.sum()
        site_p[idx] = y_prior[mask].mean(axis=0)

    hour_keys = sorted(prior_df['hour_utc'].dropna().astype(int).unique().tolist())
    hour_to_i = {hour: idx for idx, hour in enumerate(hour_keys)}
    hour_n = np.zeros(len(hour_keys), dtype=np.float32)
    hour_p = np.zeros((len(hour_keys), y_prior.shape[1]), dtype=np.float32)
    hour_series = prior_df['hour_utc'].astype(int)
    for hour in hour_keys:
        idx = hour_to_i[hour]
        mask = hour_series.values == hour
        hour_n[idx] = mask.sum()
        hour_p[idx] = y_prior[mask].mean(axis=0)

    sh_to_i = {}
    sh_n_list = []
    sh_p_list = []
    for (site, hour), group_idx in prior_df.groupby(['site', 'hour_utc']).groups.items():
        sh_to_i[(str(site), int(hour))] = len(sh_n_list)
        group_idx = np.array(list(group_idx))
        sh_n_list.append(len(group_idx))
        sh_p_list.append(y_prior[group_idx].mean(axis=0))

    sh_n = np.array(sh_n_list, dtype=np.float32)
    sh_p = np.stack(sh_p_list).astype(np.float32) if len(sh_p_list) else np.zeros((0, y_prior.shape[1]), dtype=np.float32)

    return {
        'global_p': global_p,
        'site_to_i': site_to_i,
        'site_n': site_n,
        'site_p': site_p,
        'hour_to_i': hour_to_i,
        'hour_n': hour_n,
        'hour_p': hour_p,
        'sh_to_i': sh_to_i,
        'sh_n': sh_n,
        'sh_p': sh_p,
    }


def prior_logits_from_tables(sites, hours, tables, eps=1e-4):
    n_rows = len(sites)
    p = np.repeat(tables['global_p'][None, :], n_rows, axis=0).astype(np.float32, copy=True)

    site_idx = np.fromiter((tables['site_to_i'].get(str(site), -1) for site in sites), dtype=np.int32, count=n_rows)
    hour_idx = np.fromiter((tables['hour_to_i'].get(int(hour), -1) if int(hour) >= 0 else -1 for hour in hours), dtype=np.int32, count=n_rows)
    sh_idx = np.fromiter(
        (tables['sh_to_i'].get((str(site), int(hour)), -1) if int(hour) >= 0 else -1 for site, hour in zip(sites, hours)),
        dtype=np.int32,
        count=n_rows,
    )

    valid = hour_idx >= 0
    if valid.any():
        nh = tables['hour_n'][hour_idx[valid]][:, None]
        wh = nh / (nh + 8.0)
        p[valid] = wh * tables['hour_p'][hour_idx[valid]] + (1.0 - wh) * p[valid]

    valid = site_idx >= 0
    if valid.any():
        ns = tables['site_n'][site_idx[valid]][:, None]
        ws = ns / (ns + 8.0)
        p[valid] = ws * tables['site_p'][site_idx[valid]] + (1.0 - ws) * p[valid]

    valid = sh_idx >= 0
    if valid.any():
        nsh = tables['sh_n'][sh_idx[valid]][:, None]
        wsh = nsh / (nsh + 4.0)
        p[valid] = wsh * tables['sh_p'][sh_idx[valid]] + (1.0 - wsh) * p[valid]

    np.clip(p, eps, 1.0 - eps, out=p)
    return stable_logit(p, eps=eps)


def smooth_cols_by_file(scores: np.ndarray, end_secs: np.ndarray, cols: np.ndarray, alpha: float = 0.35) -> np.ndarray:
    if alpha <= 0 or len(cols) == 0 or len(scores) <= 1:
        return scores.copy()
    order = np.argsort(end_secs)
    x = scores[order][:, cols]
    prev_x = np.concatenate([x[:1], x[:-1]], axis=0)
    next_x = np.concatenate([x[1:], x[-1:]], axis=0)
    smoothed = scores.copy()
    smoothed[np.ix_(order, cols)] = (1.0 - alpha) * x + 0.5 * alpha * (prev_x + next_x)
    return smoothed


def fuse_native_logits(logits: np.ndarray, prior_logits: np.ndarray, end_secs: np.ndarray, idx_event: np.ndarray, idx_texture: np.ndarray):
    fused = logits.copy()
    if len(idx_event):
        fused[:, idx_event] += LAMBDA_EVENT * prior_logits[:, idx_event]
    if len(idx_texture):
        fused[:, idx_texture] += LAMBDA_TEXTURE * prior_logits[:, idx_texture]
    if SMOOTH_TEXTURE > 0:
        fused = smooth_cols_by_file(fused, end_secs=end_secs, cols=idx_texture, alpha=SMOOTH_TEXTURE)
    return fused


PRIOR_TABLES = fit_prior_tables(SC_CLEAN[['site', 'hour_utc']], PRIOR_TARGETS)
print('Prior tables ready')


In [ ]:

@dataclass
class ReferenceModelConfig:
    sr: int = 32_000
    chunk_duration: float = 20.0
    n_mels: int = 224
    n_fft: int = 2048
    hop_length: int = 512
    fmin: int = 0
    fmax: int = 16_000
    top_db: float = 80.0
    power: float = 2.0
    norm: str = 'slaney'
    mel_scale: str = 'htk'
    backbone: str = 'tf_efficientnet_b0.ns_jft_in1k'
    num_classes: int = 234
    in_channels: int = 3
    dropout: float = 0.1
    drop_path_rate: float = 0.0
    gem_p_init: float = 3.0


class GEMFreqPool(nn.Module):
    def __init__(self, p_init: float = 3.0, eps: float = 1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.tensor(p_init))
        self.eps = eps

    def forward(self, x):
        p = self.p.clamp(min=1.0)
        x = x.clamp(min=self.eps).pow(p)
        x = x.mean(dim=2)
        return x.pow(1.0 / p)


class TemporalWindowHead(nn.Module):
    def __init__(self, feat_dim: int, num_classes: int, num_steps: int, dropout: float = 0.1):
        super().__init__()
        self.pre = nn.Sequential(
            nn.Conv1d(feat_dim, feat_dim, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
        )
        self.pool = nn.AdaptiveAvgPool1d(num_steps)
        self.classifier = nn.Conv1d(feat_dim, num_classes, kernel_size=1)

    def forward(self, x):
        x = self.pre(x)
        x = self.pool(x)
        windowwise_logit = self.classifier(x).transpose(1, 2)
        windowwise_prob = torch.sigmoid(windowwise_logit)
        clipwise_prob = windowwise_prob.max(dim=1).values
        return {
            'windowwise_logit': windowwise_logit,
            'windowwise_prob': windowwise_prob,
            'clipwise_prob': clipwise_prob,
        }


class MelSpectrogramTransform(nn.Module):
    def __init__(self, cfg_ref: ReferenceModelConfig):
        super().__init__()
        self.mel = T.MelSpectrogram(
            sample_rate=cfg_ref.sr,
            n_fft=cfg_ref.n_fft,
            hop_length=cfg_ref.hop_length,
            n_mels=cfg_ref.n_mels,
            f_min=cfg_ref.fmin,
            f_max=cfg_ref.fmax,
            power=cfg_ref.power,
            norm=cfg_ref.norm,
            mel_scale=cfg_ref.mel_scale,
        )
        self.db = T.AmplitudeToDB(stype='power', top_db=cfg_ref.top_db)

    @torch.no_grad()
    def forward(self, waveforms):
        waveforms = torch.nan_to_num(waveforms.float(), nan=0.0, posinf=0.0, neginf=0.0)
        mel = self.mel(waveforms)
        mel = torch.nan_to_num(mel, nan=0.0, posinf=0.0, neginf=0.0)
        mel = self.db(mel)
        mel = torch.nan_to_num(mel, nan=-80.0, posinf=0.0, neginf=-80.0)
        batch_size = mel.shape[0]
        mel_flat = mel.reshape(batch_size, -1)
        mel_min = mel_flat.min(dim=1, keepdim=True)[0].unsqueeze(-1)
        mel_max = mel_flat.max(dim=1, keepdim=True)[0].unsqueeze(-1)
        mel = (mel - mel_min) / (mel_max - mel_min + 1e-7)
        mel = torch.nan_to_num(mel, nan=0.0, posinf=1.0, neginf=0.0)
        return mel.unsqueeze(1).repeat(1, 3, 1, 1)


class WaveformLongContextSED(nn.Module):
    def __init__(self, cfg_runtime: Config, num_classes: int):
        super().__init__()
        self.ref_cfg = ReferenceModelConfig(num_classes=num_classes, chunk_duration=cfg_runtime.context_duration)
        self.mel = MelSpectrogramTransform(self.ref_cfg)
        self.backbone = timm.create_model(
            self.ref_cfg.backbone,
            pretrained=False,
            in_chans=self.ref_cfg.in_channels,
            features_only=False,
            global_pool='',
            num_classes=0,
            drop_path_rate=self.ref_cfg.drop_path_rate,
        )
        feat_dim = self.backbone.num_features
        self.freq_pool = GEMFreqPool(p_init=self.ref_cfg.gem_p_init)
        self.head = TemporalWindowHead(feat_dim, num_classes, cfg_runtime.num_output_steps, self.ref_cfg.dropout)

    def forward(self, waveforms):
        mel = self.mel(waveforms)
        features = self.backbone(mel)
        temporal = self.freq_pool(features)
        return self.head(temporal)


def extract_state_dict(ckpt):
    if isinstance(ckpt, dict):
        for key in ['model_state_dict', 'state_dict', 'model']:
            value = ckpt.get(key)
            if isinstance(value, dict):
                return value
    if isinstance(ckpt, dict):
        return ckpt
    raise TypeError('Unsupported checkpoint format')


ckpt = torch.load(CHECKPOINT_PATH, map_location='cpu')
model = WaveformLongContextSED(cfg, len(SPECIES)).to(device)
load_result = model.load_state_dict(extract_state_dict(ckpt), strict=True)
model.eval()
amp_enabled = device.type == 'cuda'

print({
    'checkpoint_epoch': int(ckpt.get('epoch', -1)) if isinstance(ckpt, dict) else None,
    'missing_keys': len(load_result.missing_keys),
    'unexpected_keys': len(load_result.unexpected_keys),
    'num_classes': len(SPECIES),
})


In [ ]:

ROW_PATTERN = re.compile(r'^(.*)_(\d+)$')
TRAIN_SC_DIR = DATA_ROOT / 'train_soundscapes'


def parse_row_id(row_id: str):
    match = ROW_PATTERN.match(str(row_id))
    if not match:
        return None, None
    return match.group(1), int(match.group(2))


def discover_test_files(data_root: Path, expected_stems: set[str]):
    search_roots = []
    for name in ['test_soundscapes', 'test_audio']:
        candidate = data_root / name
        if candidate.exists():
            search_roots.append(candidate)
    search_roots.append(data_root)

    matches = {}
    for root in search_roots:
        if not root.exists():
            continue
        for path in root.rglob('*'):
            if not path.is_file() or path.suffix.lower() not in AUDIO_SUFFIXES:
                continue
            stem = path.stem
            if stem in expected_stems and stem not in matches:
                matches[stem] = path
        if len(matches) == len(expected_stems):
            break

    return [matches[stem] for stem in sorted(matches)]


def load_soundscape(path: Path) -> np.ndarray:
    audio, _ = librosa.load(path, sr=cfg.sr, mono=True)
    return np.nan_to_num(audio, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)


def build_context_starts(expected_secs, context_duration: int, hop: int):
    if not expected_secs:
        return [0]
    max_end = int(max(expected_secs))
    last_start = max(0, max_end - context_duration)
    starts = list(range(0, last_start + 1, hop))
    if not starts:
        starts = [0]
    elif starts[-1] != last_start:
        starts.append(last_start)
    return sorted(set(int(x) for x in starts))


def slice_context(audio: np.ndarray, start_sec: int) -> np.ndarray:
    start = int(start_sec * cfg.sr)
    end = start + cfg.context_samples
    clip = audio[start:end]
    if len(clip) < cfg.context_samples:
        clip = np.pad(clip, (0, cfg.context_samples - len(clip)))
    return clip.astype(np.float32)


def build_submission(row_ids, preds, expected_ids, species):
    if len(preds) == 0:
        pred_df = pd.DataFrame(
            np.zeros((0, len(species)), dtype=np.float32),
            columns=species,
            index=pd.Index([], name='row_id'),
        )
    else:
        pred_df = pd.DataFrame(
            np.asarray(preds, dtype=np.float32),
            columns=species,
            index=pd.Index(row_ids, name='row_id'),
        )

    pred_df = pred_df[~pred_df.index.duplicated(keep='first')]

    missing_ids = [row_id for row_id in expected_ids if row_id not in pred_df.index]
    if missing_ids:
        zeros = pd.DataFrame(
            np.zeros((len(missing_ids), len(species)), dtype=np.float32),
            columns=species,
            index=pd.Index(missing_ids, name='row_id'),
        )
        pred_df = pd.concat([pred_df, zeros], axis=0)

    pred_df = pred_df.loc[expected_ids]
    return pred_df.reset_index(), len(missing_ids)


expected_ids = SAMPLE_SUB['row_id'].tolist()
expected_by_stem = {}
for row_id in expected_ids:
    stem, end_sec = parse_row_id(row_id)
    if stem is None:
        continue
    expected_by_stem.setdefault(stem, []).append(end_sec)
for stem in expected_by_stem:
    expected_by_stem[stem] = sorted(expected_by_stem[stem])

expected_stems = set(expected_by_stem)
test_files = discover_test_files(DATA_ROOT, expected_stems)
used_test_fallback = False

if len(test_files) == 0:
    fallback_files = []
    for stem in sorted(expected_stems):
        for suffix in sorted(AUDIO_SUFFIXES):
            candidate = TRAIN_SC_DIR / f'{stem}{suffix}'
            if candidate.exists():
                fallback_files.append(candidate)
                break
    test_files = fallback_files
    used_test_fallback = True
    print('Warning: hidden test files not found; using aligned train fallback for a dry run.')

matched_stems = {path.stem for path in test_files}
missing_stems = sorted(expected_stems - matched_stems)

print({
    'competition_dir': str(DATA_ROOT),
    'used_test_fallback': used_test_fallback,
    'matched_test_stems': len(matched_stems),
    'expected_test_stems': len(expected_stems),
    'missing_test_stems': len(missing_stems),
})
if test_files:
    print('First test path:', test_files[0])
if missing_stems:
    print('Missing test stems sample:', missing_stems[:5])


In [ ]:

all_row_ids = []
all_preds = []
context_duration_int = int(cfg.context_duration)
target_duration_int = int(cfg.target_duration)
hop_int = int(cfg.context_hop)

print('Running exp_008 long-context hybrid inference...')
with torch.no_grad():
    for file_idx, path in enumerate(test_files, start=1):
        stem = path.stem
        expected_secs = expected_by_stem.get(stem, [])
        if not expected_secs:
            continue

        audio = load_soundscape(path)
        context_starts = build_context_starts(expected_secs, context_duration_int, hop_int)
        expected_set = set(expected_secs)

        row_logits = {f'{stem}_{end_sec}': [] for end_sec in expected_secs}
        context_waveforms = [slice_context(audio, start_sec) for start_sec in context_starts]

        for batch_start in range(0, len(context_waveforms), cfg.batch_contexts):
            batch_audio = np.stack(context_waveforms[batch_start: batch_start + cfg.batch_contexts], axis=0)
            batch_tensor = torch.from_numpy(batch_audio).float().to(device)
            with (torch.amp.autocast(device_type='cuda', enabled=amp_enabled) if amp_enabled else nullcontext()):
                output = model(batch_tensor)
            batch_logits = output['windowwise_logit'].detach().float().cpu().numpy().astype(np.float32)

            for sample_offset, start_sec in enumerate(context_starts[batch_start: batch_start + cfg.batch_contexts]):
                for step_idx in range(cfg.num_output_steps):
                    end_sec = start_sec + target_duration_int * (step_idx + 1)
                    if end_sec not in expected_set:
                        continue
                    row_id = f'{stem}_{end_sec}'
                    row_logits[row_id].append(batch_logits[sample_offset, step_idx])

        ordered_row_ids = [f'{stem}_{end_sec}' for end_sec in expected_secs]
        raw_logits = np.zeros((len(ordered_row_ids), len(SPECIES)), dtype=np.float32)
        for row_idx, row_id in enumerate(ordered_row_ids):
            values = row_logits.get(row_id, [])
            if values:
                raw_logits[row_idx] = np.mean(np.stack(values, axis=0), axis=0)

        site, hour_utc = parse_soundscape_key(stem)
        prior_logits = prior_logits_from_tables(
            sites=np.array([site] * len(ordered_row_ids), dtype=object),
            hours=np.full(len(ordered_row_ids), hour_utc, dtype=np.int32),
            tables=PRIOR_TABLES,
        )
        fused_logits = fuse_native_logits(
            logits=raw_logits,
            prior_logits=prior_logits,
            end_secs=np.asarray(expected_secs, dtype=np.int32),
            idx_event=IDX_EVENT,
            idx_texture=IDX_TEXTURE,
        )
        fused_probs = stable_sigmoid(fused_logits)

        all_row_ids.extend(ordered_row_ids)
        all_preds.extend(fused_probs)

        if file_idx % 10 == 0 or file_idx == len(test_files):
            print(f'Processed {file_idx}/{len(test_files)} files')

submission, missing_count = build_submission(all_row_ids, all_preds, expected_ids, SPECIES)
submission.to_csv(SUBMISSION_PATH, index=False)

print('Missing row_ids filled with zeros:', missing_count)
print('Submission shape:', submission.shape)
print('Saved:', str(SUBMISSION_PATH))
print('Elapsed minutes:', round((time.time() - START) / 60.0, 2))
submission.head()


## Notes

- This notebook uses the exact winning `exp_008b` local recipe:
  - long-context `exp_008` checkpoint
  - `event + texture priors + smoothing`
- If you attach only one dataset with `fold_00/best_model.pt`, `MODEL_DATASET_HINT` can stay `None`.
- If multiple model datasets are attached, set `MODEL_DATASET_HINT` to the dataset folder name under `/kaggle/input`.
